# EquiNexus RF-DETR-Seg fixed-holdout evaluation

This run trains the Apache-2.0 RF-DETR-Seg Small model from its public COCO initialization using only the competition training partition. A grouped inner split of those 565 observations selects the checkpoint and confidence threshold. The frozen 142-observation holdout is scored once with the official scorer and never receives gradient updates or threshold selection.

The runner and frozen split live in the EquiNexus GitHub repository. This notebook uses the supplied competition input only; it does not read external MAGFiLO annotations or test labels.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

WORK = Path('/kaggle/working')
CACHE = WORK / '.cache'
for name, value in {
    'PIP_CACHE_DIR': CACHE / 'pip',
    'TORCH_HOME': CACHE / 'torch',
    'MPLCONFIGDIR': CACHE / 'matplotlib',
    'HF_HOME': CACHE / 'huggingface',
    'RF_HOME': CACHE / 'roboflow',
}.items():
    Path(value).mkdir(parents=True, exist_ok=True)
    os.environ[name] = str(value)

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'rfdetr==1.8.3',
    'pytorch-lightning==2.6.6',
    'torchmetrics[detection]==1.9.0',
    'faster-coco-eval==1.8.0',
    'supervision==0.31.6',
    'transformers==5.22.2',
    'pyDeprecate==0.9.0',
    'tensorboard==2.21.0',
], check=True)
import torch, rfdetr
print('torch', torch.__version__, 'CUDA', torch.version.cuda)
print('rfdetr', rfdetr.__version__ if hasattr(rfdetr, '__version__') else '1.8.3')
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This scheduled RF-DETR run requires a Kaggle CUDA accelerator.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/BrownAssassin/EquiNexus.git'
REPO = Path('/kaggle/working/EquiNexus')
if REPO.exists():
    raise FileExistsError(f'Refusing to overwrite {REPO}')
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print('EquiNexus source commit:', source_commit)

DATA_ROOT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/rfdetr-fixedholdout-20261112')
RUNNER = REPO / 'scripts' / 'evaluate_rfdetr_fixedholdout.py'
if not RUNNER.is_file():
    raise FileNotFoundError(RUNNER)
command = [
    sys.executable, str(RUNNER),
    '--data-root', str(DATA_ROOT),
    '--split-dir', str(REPO / 'data' / 'splits'),
    '--output-dir', str(OUTPUT),
    '--epochs', '6',
    '--resolution', '768',
]
subprocess.run(command, cwd=REPO, check=True)
print('Run outputs:', OUTPUT)
for path in sorted(OUTPUT.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT), path.stat().st_size)